In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (csm-toxin)

This notebook curates the **csm-toxin** dataset from a collection of FASTA files. Peptide sequences are parsed from multiple input files, binary labels are inferred directly from filename conventions, and duplicate consistency checks are applied before exporting a standardized dataset and metadata.

- **Toxic effect / endpoint:** toxic
- **Source:** csm-toxin
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads all FASTA-like files** (`.fasta`, `.fa`, `.faa`, `.txt`) found in the csm-toxin input directory.
- **Parses peptide sequences** using a unified FASTA reader and tracks the originating filename.
- **Infers binary labels from filenames** using simple rules:
  - filenames containing `Positive` / `positive` → `label = 1`
  - filenames containing `Negative` / `negative` → `label = 0`
  - sequences without a detectable label remain `NA`.
- **Keeps a standardized schema** with:
  - `sequence`
  - `label`
- **Checks duplicated sequences** across all input files:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description file and appends QC statistics.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv` (deduplicated dataset),
  - `detected_error_sequences.csv` (conflicting-label duplicates),
  - `metadata.json`.

In [2]:
name_source = "csm-toxin"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
input_dir = Path(PATH_INPUT) / name_source
valid_ext = {".fasta", ".fa", ".faa", ".txt"}
dfs = []

for file in input_dir.iterdir():
    if file.is_file() and file.suffix.lower() in valid_ext:
        df = read_fasta_doc(file)
        df["source_file"] = file.name
        dfs.append(df)

df_csm_toxin = pd.concat(dfs, ignore_index=True)

In [4]:
df_csm_toxin["label"] = pd.NA  # valor por defecto

mask_neg = df_csm_toxin["source_file"].str.contains(
    r"Negative|negative",
    case=False,
    regex=True,
    na=False
)

mask_pos = df_csm_toxin["source_file"].str.contains(
    r"Positive|positive",
    case=False,
    regex=True,
    na=False
)

df_csm_toxin.loc[mask_neg, "label"] = 0
df_csm_toxin.loc[mask_pos, "label"] = 1

df_csm_toxin= df_csm_toxin[["sequence", "label"]]
df_csm_toxin.shape

(236228, 2)

- Checking duplicates

In [5]:
df_csm_toxin.head()

,sequence,label
0,IALILVCWSVLSQAAQTDVEGRADKRRPIWIMGHMVNAIAQIDEFV...,1
1,MGFRVLVLVVMATTSALPFTFSEEPGRSPFRPALRSEEAQALRHGL...,1
2,MGSINLRIDDELKARSYAALEKMGVTPSEALRLMLEYIADNERLPF...,1
3,GIFSSRKCKTPSKTFKGYCTRDSNCDTSCRYEGYPAGD,1
4,MLSEEEIEYRRRDARNALASQRLEGLEPDPQVVAQMERVVVGELET...,1


In [6]:
df_csm_toxin["sequence"].unique().shape

(223039,)

In [7]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_csm_toxin, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [8]:
df_errors.shape

(1405, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_csm_toxin)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2023,
 'last update date': datetime.datetime(2023, 1, 30, 0, 0),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from uniprot;No information',
 'repository or server': 'https://bitbucket.org/ascherslab/csm-toxin/src/master/data_processing/',
 'publication': 'https://www.mdpi.com/1999-4923/15/2/431',
 'number_of_raw_sequences': 236228,
 'number_of_sequences_retained': 221634,
 'number_of_positive_sequences': 7392,
 'number_of_negative_sequences': 214242,
 'number_of_erroneous_sequences': 1405,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)